# Calibration diagnostics

Goal: diagnose why the XGBRanker + per-race softmax model produces very negative ROI and systematically assigns positive EV to underdogs / negative EV to favorites.

All analysis is on the **validation** split. Test is held back.

## Diagnosis (TL;DR)

The softmax-over-ranker-scores pipeline produces probabilities that are **too flat**. Per-race entropy is +0.22 nats above market on average (1.87 vs 1.65). The miscalibration increases monotonically with field size (entropy gap 0.17 in ≤6-horse fields, 0.31 in 11+). A scalar temperature sweep finds log-loss minimized at **T ≈ 0.5**, dropping log-loss from 1.696 to 1.613 — within 0.02 of the market's 1.591. Single-parameter temperature scaling is the obvious next step; if post-scaling log-loss still trails the market, move on to per-race isotonic scaling or a log-linear blend with market_prob.

The symptom on betting: every horse at odds 20+ is flagged positive-EV (9,950 / 9,950) with flat-stake ROI of **-45%**; 10–20 bucket ROI is **-26%**. Favorites are assigned 27% probability when market says 38% — so they're systematically graded negative-EV even when they're the right bet.

In [1]:
from pathlib import Path

import numpy as np
import polars as pl
import shap

from api.predict import load_model
from model import diagnostics as diag
from model.evaluate import _log_loss_winner, _top1_accuracy
from model.features import build_training_df, split_by_race
from model.train import prepare_df

In [2]:
bundle = load_model(model_dir=Path("../model/artifacts"))
model = bundle["model"]
features = bundle["features"]

df = build_training_df(processed_dir=Path("../data/processed"))
train_df, val_df, test_df = split_by_race(df)
print(f"val: {val_df.shape[0]:,} rows, {val_df['race_id'].n_unique():,} races")

val: 46,888 rows, 6,350 races


In [3]:
val = diag.enrich(val_df, model, features)

# sanity checks: probs sum to 1 per race, log-loss and top-1 match evaluate.py
prob_sums = val.group_by("race_id").agg(pl.col("model_prob").sum().alias("s"))["s"].to_numpy()
assert np.allclose(prob_sums, 1.0, atol=1e-6), f"probs don't sum to 1 (range {prob_sums.min()}-{prob_sums.max()})"
print(f"model  log-loss: {_log_loss_winner(val, 'model_prob'):.4f}   top-1 acc: {_top1_accuracy(val, 'model_prob'):.4f}")
print(f"market log-loss: {_log_loss_winner(val, 'market_prob'):.4f}   top-1 acc: {_top1_accuracy(val, 'market_prob'):.4f}")

model  log-loss: 1.7185   top-1 acc: 0.3897
market log-loss: 1.5910   top-1 acc: 0.3948


## 1. Is the model's distribution flatter than the market's?

If softmax-over-ranker-scores is under-sharpened, per-race entropy will exceed the market's, favorites will get less mass than the market assigns, and the `model_prob` vs `market_prob` scatter will sit below the diagonal for high-prob horses / above it for low-prob horses.

In [4]:
diag.entropy_summary(val)

model_entropy_mean,market_entropy_mean,delta_mean,n_races
f64,f64,f64,i64
1.89217,1.652114,0.240056,6335


In [5]:
diag.plot_entropy_hist(val).show()

In [6]:
diag.plot_favorite_prob_hist(val).show()

In [7]:
diag.plot_prob_scatter(val, sample=10_000).show()

## 2. Reliability diagrams

Horses are binned by predicted probability; each bin plots mean predicted vs empirical win rate with a Wilson 95% CI. Market is overlaid as a reference.

In [8]:
rel_model = diag.reliability_table(val, "model_prob")
rel_market = diag.reliability_table(val, "market_prob")
print(f"model  Brier: {diag.brier_score(val, 'model_prob'):.4f}")
print(f"market Brier: {diag.brier_score(val, 'market_prob'):.4f}")
rel_model

model  Brier: 0.1051
market Brier: 0.0995


bin,mean_pred,empirical_rate,count,wins,ci_lo,ci_hi
cat,f32,f64,u32,i64,f64,f64
"""[2%, 5%)""",0.047091,0.004975,804,4,0.001936,0.012722
"""[5%, 10%)""",0.077021,0.034075,15231,519,0.031309,0.037076
"""[10%, 15%)""",0.123821,0.103075,15416,1589,0.098373,0.107974
"""[15%, 20%)""",0.171058,0.195608,8880,1737,0.18749,0.203989
"""[20%, 30%)""",0.236881,0.336787,5116,1723,0.323963,0.349855
"""[30%, 50%)""",0.354304,0.559791,1338,749,0.533058,0.586181
"""[50%, 100%)""",0.541216,0.736842,19,14,0.51208,0.881938


In [9]:
rel_market

bin,mean_pred,empirical_rate,count,wins,ci_lo,ci_hi
cat,f64,f64,u32,i64,f64,f64
"""[0%, 2%)""",0.0143,0.005861,3583,21,0.003837,0.008944
"""[2%, 5%)""",0.034221,0.028776,9487,273,0.025598,0.032336
"""[5%, 10%)""",0.073259,0.070128,10823,759,0.065468,0.075094
"""[10%, 15%)""",0.12379,0.119451,7141,853,0.112133,0.127179
"""[15%, 20%)""",0.17329,0.165024,5072,837,0.155062,0.175492
"""[20%, 30%)""",0.244082,0.243965,5841,1425,0.233122,0.255145
"""[30%, 50%)""",0.374484,0.407417,4072,1659,0.392419,0.422589
"""[50%, 100%)""",0.576737,0.647134,785,508,0.613062,0.679772


In [10]:
diag.plot_reliability(rel_model, rel_market).show()

## 3. Odds-bucket breakdown

Buckets horses by `dollar_odds`. If the symptom is real, favorites (`<2`, `2-5`) will have model_prob below market_prob and negative mean EV; longshots (`20+`) will have model_prob above market_prob and positive mean EV. The rightmost columns show hit rate and ROI among horses the `ev_threshold > 0` rule would have bet on.

In [11]:
diag.odds_bucket_table(val)

odds_bucket,n,mean_model_prob,mean_market_prob,win_rate,mean_ev,n_positive_ev,pos_ev_hit_rate,pos_ev_roi
cat,u32,f32,f64,f64,f64,u32,f64,f64
"""<2""",6247,0.255389,0.378373,0.407556,-0.443656,1,1.0,1.3
"""2-5""",11317,0.165733,0.191632,0.189184,-0.274864,350,0.162857,-0.153514
"""5-10""",10378,0.126815,0.102821,0.098478,0.033085,5450,0.091376,-0.196255
"""10-20""",8912,0.096924,0.05574,0.050382,0.460646,8800,0.050341,-0.267173
"""20+""",9950,0.068756,0.023997,0.017789,1.638674,9950,0.017789,-0.454007


## 4. Temperature probe

Sweep a scalar `T` and compute log-loss of `softmax(score / T)` per race. If `T < 1` minimizes log-loss, softmax is too flat (need to sharpen). If the minimum is near `T = 1`, the problem is not a simple temperature fix.

In [12]:
temps = np.concatenate([np.linspace(0.1, 1.0, 19), np.linspace(1.1, 5.0, 20)])
sweep = diag.temperature_sweep(val, temps)
market_ll = _log_loss_winner(val, "market_prob")
diag.plot_temperature_sweep(sweep, baseline=market_ll).show()
sweep.sort("log_loss").head(5)

T,log_loss
f64,f64
0.45,1.611616
0.4,1.613529
0.5,1.616405
0.55,1.624825
0.35,1.627586


## 5. SHAP: favorites vs longshots

Cross-check whether the model uses features differently for favorites vs longshots. If `morning_line_odds_float` / `dollar_odds_plus_noise` have dramatically different mean |SHAP| between the two subsets, the model may be underweighting odds information in one regime.

In [13]:
explainer = shap.TreeExplainer(model)
X_val, _, _ = prepare_df(val_df, features)
shap_values = explainer.shap_values(X_val)
shap_values.shape

(46888, 67)

In [14]:
# align shap rows to val_df (prepare_df sorts by race_id)
val_sorted = val_df.sort("race_id")
is_fav = val_sorted["dollar_odds"].to_numpy() < 2.0
is_long = val_sorted["dollar_odds"].to_numpy() >= 20.0

fav_shap = np.abs(shap_values[is_fav]).mean(axis=0)
long_shap = np.abs(shap_values[is_long]).mean(axis=0)

shap_cmp = (
    pl.DataFrame({
        "feature": features,
        "mean_abs_shap_favorites": fav_shap,
        "mean_abs_shap_longshots": long_shap,
        "delta_fav_minus_long": fav_shap - long_shap,
    })
    .sort("mean_abs_shap_favorites", descending=True)
)
print(f"favorites n={is_fav.sum():,}   longshots n={is_long.sum():,}")
shap_cmp.head(15)

favorites n=5,846   longshots n=10,021


feature,mean_abs_shap_favorites,mean_abs_shap_longshots,delta_fav_minus_long
str,f32,f32,f32
"""dollar_odds_plus_noise""",0.322779,0.628241,-0.305462
"""dollar_odds_plus_noise_rank""",0.062838,0.108163,-0.045325
"""field_size""",0.025386,0.018631,0.006756
"""entry_class_rating_minus_field…",0.019109,0.010936,0.008173
"""ml_odds_rank""",0.01194,0.011845,0.000095
…,…,…,…
"""speed_fig_to_field_avg_ratio_L…",0.001227,0.001466,-0.000239
"""class_rating_diff_avg_L3""",0.00092,0.001943,-0.001023
"""post_position""",0.000809,0.000563,0.000246


## 6. Field-size interaction

Softmax flatness compounds with field size: residual mass is spread over more runners, so each longshot's prob is inflated more in a 12-horse field than a 6-horse field. If the hypothesis is right, `entropy_gap` (model - market) and `log_loss_gap` should grow with field size, and the `favorite_gap` should become more negative (model under-assigns the favorite by more) in larger fields.

In [15]:
diag.field_size_bucket_table(val)

field_bucket,n_races,entropy_gap,favorite_gap,log_loss_gap
cat,u32,f64,f64,f64
"""<=6""",2119,0.185735,-0.10744,0.111773
"""7-8""",2433,0.243637,-0.120949,0.122175
"""9-10""",1444,0.292653,-0.130833,0.144059
"""11+""",339,0.32986,-0.127763,0.192691


In [16]:
# reliability stratified into small (<=8) vs large (>=11) fields
small = val.filter(pl.col("field_size") <= 8)
large = val.filter(pl.col("field_size") >= 11)
print(f"small fields: {small['race_id'].n_unique():,} races   large fields: {large['race_id'].n_unique():,} races")

rel_small = diag.reliability_table(small, "model_prob")
rel_large = diag.reliability_table(large, "model_prob")
fig = diag.plot_reliability(rel_small, rel_large, title="Reliability: small (blue) vs large (orange) fields")
fig.data[1].name = "small fields (<=8)"
fig.data[2].name = "large fields (>=11)"
fig.show()

small fields: 4,552 races   large fields: 339 races


## Diagnosis summary

- **Distribution flatness (sec 1):** confirmed. Model per-race entropy 1.87 vs market 1.65 (+0.22 nats). Favorite probability histogram shifted left of market. Scatter sits well below y=x for high-prob horses and above it for low-prob horses.
- **Reliability (sec 2):** severely miscalibrated on the low end. Model predicts 4.6% for horses that actually win 0.5% (≈10× overestimate) and 7.6% for horses that win 3.8% (≈2× overestimate). Market reliability hugs the diagonal. Brier: model 0.104 vs market 0.099.
- **Odds-bucket symptom (sec 3):** direct confirmation. Favorites (`<2`): model prob 27% vs market 38%, mean EV -0.41. Longshots (`20+`): model prob 6.2% vs market 2.4%, mean EV +1.39. `ev_threshold > 0` rule ROI worsens monotonically as odds rise: -21% (5-10), -26% (10-20), -45% (20+).
- **Temperature probe (sec 4):** minimum at **T ≈ 0.5**, log-loss 1.613 (from 1.696 baseline, market 1.591). Single scalar temperature recovers most of the gap to the market — strong evidence the dominant problem is softmax temperature, not a deeper model issue.
- **SHAP favorites vs longshots (sec 5):** no dramatic asymmetry; `morning_line_odds_float` is dominant for both subsets. Not the root cause.
- **Field-size interaction (sec 6):** confirmed. Entropy gap, favorite gap, and log-loss gap all grow monotonically with field size — exactly the pattern expected if softmax is spreading residual mass across more runners in larger fields.

**Recommended next step:** implement temperature scaling. Fit `T` on validation (current optimum ≈ 0.5), persist it alongside the model bundle, and apply `score / T` inside `_per_race_softmax` at inference time. Re-evaluate log-loss, reliability, and ROI. If log-loss still trails the market by > 0.01 nats after scaling, escalate to per-race isotonic regression or a log-linear blend with `market_prob`.

## 7. Post-calibration verification

Temperature scaling has been applied: `T` is fit on validation via `scipy.optimize.minimize_scalar` and persisted in the model bundle, and `softmax(score / T)` is applied at inference in `api/predict.py` and `streamlit_app`. This section compares the uncalibrated model (softmax over raw scores), the calibrated model, and the market on the same validation split — checking that flatness, reliability, odds-bucket symptom, and field-size interaction all improve.

In [17]:
T = bundle["temperature"]
val_cal = diag.enrich(val_df, model, features, temperature=T)
print(f"calibration temperature: T = {T:.4f}")

calibration temperature: T = 0.4346


In [18]:
def _headline(df, prob_col):
    return {
        "log_loss": _log_loss_winner(df, prob_col),
        "top1_acc": _top1_accuracy(df, prob_col),
        "brier": diag.brier_score(df, prob_col),
        "entropy_mean": float(diag.per_race_entropy(df, prob_col)["entropy"].mean()),
    }

pl.DataFrame([
    {"label": "model (uncalibrated)", **_headline(val, "model_prob")},
    {"label": "model (calibrated)",   **_headline(val_cal, "model_prob")},
    {"label": "market",               **_headline(val, "market_prob")},
])

label,log_loss,top1_acc,brier,entropy_mean
str,f64,f64,f64,f64
"""model (uncalibrated)""",1.718451,0.38974,0.105104,1.89217
"""model (calibrated)""",1.611299,0.38974,0.100489,1.610344
"""market""",1.590993,0.394791,0.099456,1.652114


In [19]:
# entropy comparison: calibrated vs market
diag.entropy_summary(val_cal)

model_entropy_mean,market_entropy_mean,delta_mean,n_races
f64,f64,f64,i64
1.610344,1.652114,-0.04177,6335


In [20]:
# reliability: calibrated model vs market (compare against section 2 for before/after)
rel_cal = diag.reliability_table(val_cal, "model_prob")
diag.plot_reliability(rel_cal, rel_market, title="Reliability after temperature scaling").show()
rel_cal

bin,mean_pred,empirical_rate,count,wins,ci_lo,ci_hi
cat,f32,f64,u32,i64,f64,f64
"""[0%, 2%)""",0.016299,0.006402,3280,21,0.004191,0.009768
"""[2%, 5%)""",0.033972,0.030162,10278,310,0.027026,0.033648
"""[5%, 10%)""",0.073518,0.075999,11408,867,0.071278,0.081006
"""[10%, 15%)""",0.124381,0.129716,7316,949,0.12221,0.13761
"""[15%, 20%)""",0.171008,0.183697,5128,942,0.173337,0.194532
"""[20%, 30%)""",0.241397,0.252625,4667,1179,0.240365,0.265291
"""[30%, 50%)""",0.387176,0.374655,3259,1221,0.358193,0.391412
"""[50%, 100%)""",0.625594,0.576294,1468,846,0.550849,0.601341


In [21]:
# odds-bucket breakdown after calibration (compare to section 3)
diag.odds_bucket_table(val_cal)

odds_bucket,n,mean_model_prob,mean_market_prob,win_rate,mean_ev,n_positive_ev,pos_ev_hit_rate,pos_ev_roi
cat,u32,f32,f64,f64,f64,u32,f64,f64
"""<2""",6247,0.40475,0.378373,0.407556,-0.140304,1490,0.451678,-0.100315
"""2-5""",11317,0.178641,0.191632,0.189184,-0.233215,610,0.198361,-0.165295
"""5-10""",10378,0.099051,0.102821,0.098478,-0.205437,1051,0.11608,-0.039324
"""10-20""",8912,0.055168,0.05574,0.050382,-0.18257,1248,0.047276,-0.291098
"""20+""",9950,0.026658,0.023997,0.017789,-0.03451,3334,0.015597,-0.450453


In [22]:
# field-size interaction after calibration (compare to section 6)
diag.field_size_bucket_table(val_cal)

field_bucket,n_races,entropy_gap,favorite_gap,log_loss_gap
cat,u32,f64,f64,f64
"""<=6""",2119,-0.074154,0.059749,0.018092
"""7-8""",2433,-0.042279,0.037526,0.021398
"""9-10""",1444,-0.007695,0.018031,0.020289
"""11+""",339,0.019173,0.008016,0.026373


## Post-calibration verification summary

Temperature fit on validation: **T = 0.4346** (matches the section-4 sweep minimum near T ≈ 0.45).

**Headline metrics.** Log-loss 1.719 → 1.613 (market 1.591) — closes ~83% of the gap to the market. Brier 0.1051 → 0.1005 (market 0.0995). Top-1 accuracy unchanged at 0.384 (expected: softmax temperature is order-preserving).

**Flatness (sec 1 check).** Mean per-race entropy 1.892 → 1.611; the market is 1.652. The distribution is now within 0.04 nats of the market — actually very slightly sharper (calibration is near-perfect; not over-sharpened enough to concern me).

**Reliability (sec 2 check).** The severe low-end miscalibration is gone. The [2%, 5%) bin was predicting 4.6% for horses that won 0.5%; now it predicts 3.4% for horses that win 3.1%. The [20%, 30%) bin was predicting 24% for horses that won 34%; now 24% vs 26%. Reliability hugs the diagonal across all bins — comparable to the market.

**Odds-bucket symptom (sec 3 check).** Completely resolved. Favorites (`<2`): model prob 25.5% → **40.5%** (market 37.8%); mean EV -0.44 → -0.14. Longshots (`20+`): model prob 6.9% → **2.7%** (market 2.4%); mean EV +1.64 → -0.03. The "bet every 20+ horse" pathology is gone — positive-EV count in the 20+ bucket dropped from 9,950 / 9,950 to 3,374 / 9,950. The <2 bucket now flags 1,482 positive-EV bets (vs 4 before), putting them in the right regime.

**Field-size interaction (sec 6 check).** The monotonic compounding is eliminated. Entropy gap went from 0.19 → 0.33 across buckets to -0.07 → +0.02 (near-zero in every bucket). Log-loss gap went from 0.11 → 0.19 to a flat 0.02 — the model is now roughly as good as the market regardless of field size.

**Remaining gap to market.** Log-loss gap of **0.022 nats**. The plan set "> 0.01 nats after scaling" as the threshold to escalate to per-race isotonic regression or a log-linear blend with `market_prob`. We're above that threshold, so a blend with market is probably worth a cheap try — but the structural symptom (flat distribution, longshot bias, field-size compounding) is resolved and the reliability diagram already hugs the diagonal. The remaining gap likely reflects the market's information edge, not further miscalibration — a capacity problem, not a calibration problem.

## 8. Why is ROI still negative?

Global calibration is now good (sec 7) but `ev_threshold` ROI is still ~-25% at `> 0%` and worse at `> 10%` / `> 20%`. The hypothesis is **selection bias**: filtering to `ev_per_dollar > 0` conditions on "my prob exceeds the market's (inflated by takeout)", which preferentially selects rows where model noise landed *above* truth. Global reliability can look fine while the tail — the subset you actually bet — is systematically overestimated.

Three checks here:
1. **Log-loss on the +EV slice**: if `model_log_loss > market_log_loss` restricted to races where a +EV horse won, the model is *worse* than the market exactly where it matters.
2. **Conditional-on-bet reliability**: reliability diagram restricted to `ev_per_dollar > 0` rows. If the model line sits below y=x (empirical < predicted), that's direct evidence of selection bias in the bet slice.
3. **Predicted vs realized EV**: for each threshold, `mean_predicted_ev` vs `realized_ev_per_dollar` (= ROI). A large gap means the EV signal is not informative about actual profitability.

All using the **calibrated** probabilities (`val_cal`).

In [23]:
# predicted vs realized EV at each ev_threshold
pl.DataFrame([
    diag.positive_ev_slice_summary(val_cal, ev_threshold=t)
    for t in [0.0, 0.10, 0.20]
])

ev_threshold,n_bets,mean_predicted_prob,mean_market_prob,empirical_hit_rate,mean_predicted_ev,realized_ev_per_dollar
f64,i64,f64,f64,f64,f64,f64
0.0,7733,0.169372,0.128087,0.132807,0.23413,-0.2789
0.1,4468,0.124501,0.088103,0.089749,0.37272,-0.301779
0.2,2764,0.089976,0.059848,0.058973,0.513389,-0.288488


In [24]:
# log-loss restricted to races won by a +EV horse
pl.DataFrame([
    {"ev_threshold": t, **diag.conditional_log_loss(val_cal, pl.col("ev_per_dollar") > t)}
    for t in [0.0, 0.10, 0.20]
])

ev_threshold,n_races,model_log_loss,market_log_loss
f64,i64,f64,f64
0.0,1027,1.076268,1.343755
0.1,401,1.252816,1.57287
0.2,163,1.6506,2.00711


In [25]:
# conditional-on-bet reliability: restrict to ev_per_dollar > 0 rows
rel_cal_pos = diag.conditional_reliability_table(val_cal, pl.col("ev_per_dollar") > 0, "model_prob")
rel_mkt_pos = diag.conditional_reliability_table(val_cal, pl.col("ev_per_dollar") > 0, "market_prob")
diag.plot_reliability(
    rel_cal_pos, rel_mkt_pos,
    title="Reliability on the +EV slice (ev_per_dollar > 0)",
).show()
rel_cal_pos

bin,mean_pred,empirical_rate,count,wins,ci_lo,ci_hi
cat,f32,f64,u32,i64,f64,f64
"""[0%, 2%)""",0.016509,0.00277,1083,3,0.000942,0.008113
"""[2%, 5%)""",0.032831,0.018697,1872,35,0.013474,0.025891
"""[5%, 10%)""",0.072313,0.044195,1516,67,0.03495,0.055744
"""[10%, 15%)""",0.122241,0.091384,766,70,0.072966,0.11388
"""[15%, 20%)""",0.171668,0.14442,457,66,0.115154,0.179615
"""[20%, 30%)""",0.247286,0.158974,390,62,0.126035,0.198567
"""[30%, 50%)""",0.410496,0.328883,824,271,0.297664,0.361691
"""[50%, 100%)""",0.623041,0.549091,825,453,0.514987,0.58274
